In [22]:
import os
import getpass

os.environ["GOOGLE_GEMINI_API_KEY"] = getpass.getpass("Gemini API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")


In [20]:
import google.generativeai  as genai

# genai.configure(api_key="YOUR_GOOGLE_GEMINI_API_KEY")

models = genai.list_models()

for model in models:
    if model.name.startswith("models/gemini-2.0-flash"):
        print(model.name, model.supported_generation_methods)


models/gemini-2.0-flash-exp ['generateContent', 'countTokens', 'bidiGenerateContent']
models/gemini-2.0-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-preview-02-05 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-preview ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-thinking-exp-01-21 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-thinking-exp ['generateContent', 'countTokens', 'createCachedContent', 'batchGe

In [38]:
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image

# Load Stable Diffusion
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5"
).to("cuda" if torch.cuda.is_available() else "cpu")

# prompt = "A chef in a rustic kitchen kneading bread dough, cinematic lighting, 85mm lens, hyper-realistic, 16:9"
prompt = ('A chef in a sunlit rustic kitchen, flour dust dancing in golden morning light streaming through vintage windows, hands kneading artisanal bread dough on a worn wooden table scattered with fresh herbs and heirloom tomatoes, copper pots gleaming in background, shot with 85mm lens, shallow depth of field, warm color grading, professional food photography style, hyper-realistic detail --aspect 16:9')

image = pipe(prompt).images[0]

image.save("'images/chef_sdp.png")
image.show()


ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [32]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import base64
from openai import OpenAI

# Google
client = genai.Client()

# OpenAI
# client = OpenAI()

contents = ('A chef in a sunlit rustic kitchen, flour dust dancing in golden morning light streaming through vintage windows, hands kneading artisanal bread dough on a worn wooden table scattered with fresh herbs and heirloom tomatoes, copper pots gleaming in background, shot with 85mm lens, shallow depth of field, warm color grading, professional food photography style, hyper-realistic detail --aspect 16:9')

response = client.models.generate_images(
    model='imagen-3.0-generate-002',
    prompt=contents,
    config=types.GenerateImagesConfig(
        number_of_images= 1,
    )
)

# deprecated!!!
# response = client.models.generate_content(
#     model="gemini-2.0-flash-preview-image-generation",
#     contents=contents,
#     config=types.GenerateContentConfig(
#       response_modalities=['TEXT', 'IMAGE']
#     )
# )

# Open AI - requires verification before this could work
# response = client.images.generate(
#     # model="gemini-2.0-flash-preview-image-generation",
#     model="gpt-image-1",
#     prompt=contents,
#     size="1024x1024"
# )

# for part in response.candidates[0].content.parts:
#   if part.text is not None:
#     print(part.text)
#   elif part.inline_data is not None:
#     image = Image.open(BytesIO((part.inline_data.data)))
#     image.save('images/chef_openai.png')
#     image.show()

for generated_image in response.generated_images:
  generated_image = Image.open(BytesIO(generated_image.image.data))
  generated_image.save('images/chef_generated.png')
  generated_image.image.show()

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Imagen API is only accessible to billed users at this time.', 'status': 'INVALID_ARGUMENT'}}

In [41]:
os.environ["STABILITY_API_KEY"] = getpass.getpass("Stability AI API Key:")


In [ ]:
import stability_sdk.interfaces.gooseai.generation.generation_pb2 as generation
from stability_sdk import client
from PIL import Image
import io


stability_api = client.StabilityInference(
    key=os.environ["STABILITY_API_KEY"],
    verbose=True,
)

prompt = ('A chef in a sunlit rustic kitchen, flour dust dancing in golden morning light streaming through vintage windows, hands kneading artisanal bread dough on a worn wooden table scattered with fresh herbs and heirloom tomatoes, copper pots gleaming in background, shot with 85mm lens, shallow depth of field, warm color grading, professional food photography style, hyper-realistic detail --aspect 16:9')

answers = stability_api.generate(prompt=prompt)

for resp in answers:
    for artifact in resp.artifacts:
        if artifact.type == generation.ARTIFACT_IMAGE:
            image = Image.open(io.BytesIO(artifact.binary))
            image.save("images/chef_stability.png")
            image.show()


<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Task #1: Adjust the prompt to include face of the chef.

### Answer:

I used Stability AI API (as the given Google API model has been deprecated) to generate the image and it actually generated the chef with the entire face. Please see chef_stability.png for the generated image under \images folder.

### Insights: Share any insights you have learned by playing around with the prompt.

### Answer:

Insights from Experimenting with the Prompt

1. Prompt Length Matters

    The prompt used is highly descriptive, which is great for photorealism. However, some models (like Stable Diffusion or Gemini) tend to prioritize earlier tokens more heavily.

    To have more focus on the chef and kneading bread, those details can be moved at the start.

    For example:
        “A chef kneading artisanal bread dough on a rustic wooden table, sunlight streaming through vintage windows, flour dust floating in the air, heirloom tomatoes scattered, copper pots gleaming in the background…”

    This matters because models “pay more attention” to the first ~50 tokens.

2. Camera Angles & Lenses Change the Mood

    “shot with 85mm lens” is used which is perfect for portraits and close-ups.

    For wider context of the kitchen, “35mm lens” or “24mm lens” can be tried.

    Example:

        85mm → more bokeh, shallow depth, focus on hands.
        35mm → shows kitchen ambiance better.
        24mm → gives cinematic wide-angle vibes.

3. Lighting Keywords Are Powerful

    “Golden morning light” already sets the tone, but exposure can be guided by using:

    “Soft diffused light” → natural, airy photography.
    “Hard shadows” → dramatic, cinematic style.
    “Backlit” → emphasizes flour dust beautifully.

4. Use Explicit Style Keywords

    For better results on top of including “professional food photography style”, stack styles such as the following can be used:

    “Hyper-realistic, HDR, 8K, magazine-style food photography”
    “Cinematic color grading, volumetric lighting”

5. The Aspect Ratio Flag

    Aspect 16:9, which is perfect for landscape, cinematic shots.

    But:

    For social posts → try --aspect 4:5.
    For banners or thumbnails → --aspect 21:9.

6. Importance of Composition Terms

    Adding keywords like these can improve framing:

    “Rule of thirds” → more balanced images.
    “Depth of field” → increases realism.
    “Over-the-shoulder shot” → immersive perspective.

</div>

### Text and Image -> Image

In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO

import PIL.Image

client = genai.Client()


image = PIL.Image.open('images/brad.jpg')

# Convert the image to a base64 string
buffered = BytesIO()
image.save(buffered, format="JPEG")
image_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')


text_input = 'This is a picture of me. Make me wear a crown like a king.'

content = types.Content(
  role="user",
  parts=[
    types.Part(text=text_input),
    types.Part.from_bytes(
        mime_type="image/jpeg",
        data=base64.b64decode(image_base64)
    )
  ]
)

### Deprecated at the time of executing the code ###
response = client.models.generate_content(
    model="models/gemini-2.0-flash-preview-image-generation",
    contents=content,
    config=types.GenerateContentConfig(
      response_modalities=['TEXT', 'IMAGE']
    )
)

for part in response.candidates[0].content.parts:
  if part.text is not None:
    print(part.text)
  elif part.inline_data is not None:
    image = Image.open(BytesIO((part.inline_data.data)))
    image.save('images/brad_crown.jpg')
    image.show()

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2.0-flash-preview-image-generation is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [35]:
from pprint import pprint

pprint(response)

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""I will add a regal gold crown adorned with shimmering jewels to your head, transforming your portrait into that of a distinguished king.

"""
          ),
          Part(
            inline_data=Blob(
              data=<... Max depth ...>,
              mime_type=<... Max depth ...>
            )
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.0-flash-preview-image-generation',
  response_id='Tz-paKivEuOIz7IPsenCyAU',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=1315,
    candidates_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.IMAGE: 'IMAGE'>,
        token_count=1290
      ),
    ]

<div style="background-color: #204B8E; color: white; padding: 10px; border-radius: 5px;">

### Task #1: Analyze the structure of the response. why is it structured this way?

### Answer:

The GenerateContentResponse is a structured container returned by the model that organizes its output, metadata, and usage information. At the top level, it includes fields like automatic_function_calling_history, candidates, model_version, and total_token_count. The candidates array holds one or more Candidate objects, each containing content (with a role and a list of parts), a finish_reason indicating why generation stopped, and an index. Each Part can contain text and optional binary data (inline_data as a Blob with data and mime_type), enabling the coexistence of text, images, or other media. This hierarchical structure allows for multiple alternative responses, mixed media support, standardized parsing, and inclusion of detailed metadata such as model version and token usage, making it flexible, extensible, and predictable for client applications.

</div>

In [34]:
from google import genai
from google.genai import types

client = genai.Client()

# Upload the first image
image1_path = "images/brad.jpg"
uploaded_file = client.files.upload(file=image1_path)

# Prepare the second image as inline data
image2_path = "images/brad_crown.jpg"
with open(image2_path, 'rb') as f:
    img2_bytes = f.read()

# Create the prompt with text and multiple images
response = client.models.generate_content(

    model="gemini-2.5-flash",
    contents=[
        "What is different between these two images?",
        uploaded_file,  # Use the uploaded file reference
        types.Part.from_bytes(
            data=img2_bytes,
            mime_type='image/png'
        )
    ]
)

print(response.text)

The main difference between the two images is that in the second image, the man is wearing a **crown** on his head, while in the first image, he is not.


### Video processing

In [45]:
client = genai.Client()
response = client.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=types.Content(
        parts=[
            types.Part(
                file_data=types.FileData(file_uri='https://www.youtube.com/watch?v=G32jFMsS7ow&t=1s')
            ),
            types.Part(text='Please summarize the video in 3 sentences.')
        ]
    )
)

In [56]:
from pprint import pprint
pprint(response.text)

('Multimodal AI is the next generation of generative AI, built around natural '
 'human interfaces like voice, vision, and sound, unlike traditional '
 'keyboards. The video introduces two approaches to conversational AI: the '
 'sequential approach (TTS-LLM-STT), which involves multiple steps like '
 'transcribing speech to text and then converting a text response back to '
 'speech, often causing delays. In contrast, the multimodal approach directly '
 'processes raw audio and visual data into "embeddings," allowing for blazing '
 'fast, natural-sounding responses because the AI "thinks in sound, not '
 'words," though it\'s currently harder to control mid-process for safety.')


### Audio processing

In [58]:
from google.genai import types

with open('audio/a_projectile_is.wav', 'rb') as f:
    audio_bytes = f.read()

response = client.models.generate_content(
  model='gemini-2.5-flash',
  contents=[
    'transcribe',
    types.Part.from_bytes(
      data=audio_bytes,
      mime_type='audio/mp3',
    )
  ]
)

pprint(response.text)

('A projectile is any object that is thrown or projected into the air and is '
 'only subject to the force of gravity. Projectile motion is the specific type '
 'of motion that a projectile undergoes, following a curved path.')


In [59]:
response = client.models.generate_content(
  model='gemini-2.5-flash',
  contents=[
    'describe this audio clip',
    types.Part.from_bytes(
      data=audio_bytes,
      mime_type='audio/mp3',
    )
  ]
)

pprint(response.text)

('The audio clip features a clear, **male voice** speaking in a calm, '
 'informative, and instructional tone. The speaker is explaining scientific '
 'concepts related to **physics**, specifically defining **"projectile"** and '
 '**"projectile motion"** and their relationship to **gravity**.\n'
 '\n'
 'The speech is articulate and easy to understand, with no discernible '
 'background noise. It sounds like an excerpt from an educational or '
 'explanatory segment.')
